In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achooadore/pathvqa-rag/pathvqa_fusion.index
/kaggle/input/datasets/achooadore/pathvqa-rag/pathvqa_metadata.pt
/kaggle/input/datasets/achooadore/pathvqa-rag/pathvqa_gated_fusion.pth


In [2]:
!pip install -U transformers
!pip install -q --upgrade accelerate
!pip install -q faiss-cpu
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.3 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 7.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.

In [3]:
import os
import torch
import time
import evaluate
import numpy as np
import gc
import random
import faiss
import open_clip
import torch.nn as nn
import transformers
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from PIL import ImageFile
from transformers import AutoModel, AutoTokenizer

# --- 1. CẤU HÌNH & VÁ LỖI KIẾN TRÚC ---
transformers.PreTrainedModel.all_tied_weights_keys = {}
transformers.logging.set_verbosity_error()
ImageFile.LOAD_TRUNCATED_IMAGES = True

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

saveDir = "/kaggle/input/datasets/achooadore/pathvqa-rag"

# --- 2. KIẾN TRÚC GATED FUSION ---
class GatedFusion(nn.Module):
    def __init__(self, embedDim=512):
        super(GatedFusion, self).__init__()
        self.gateLayer = nn.Linear(embedDim * 2, embedDim)
        self.sigmoidFunc = nn.Sigmoid()
        self.fusionLayer = nn.Linear(embedDim * 2, embedDim)

    def forward(self, imageEmbeds, textEmbeds):
        combinedEmbeds = torch.cat((imageEmbeds, textEmbeds), dim=1)
        gateValue = self.sigmoidFunc(self.gateLayer(combinedEmbeds))
        gatedImage = imageEmbeds * gateValue
        gatedText = textEmbeds * (1 - gateValue)
        fusedOutput = self.fusionLayer(torch.cat((gatedImage, gatedText), dim=1))
        fusedOutput = fusedOutput / fusedOutput.norm(dim=-1, keepdim=True)
        return fusedOutput

# --- 3. RAG RETRIEVER (PATHVQA) ---
class PathVqaRetriever:
    def __init__(self, saveDirectory=saveDir):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        
        self.faissIndex = faiss.read_index(f"{saveDirectory}/pathvqa_fusion.index")
        self.metaDataList = torch.load(f"{saveDirectory}/pathvqa_metadata.pt")
        
        self.fusionModel = GatedFusion(embedDim=512).to(self.device)
        self.fusionModel.load_state_dict(torch.load(f"{saveDirectory}/pathvqa_gated_fusion.pth", map_location=self.device))
        self.fusionModel.eval()
        
        modelName = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        self.bioMedClip, _, self.preprocessImage = open_clip.create_model_and_transforms(modelName)
        self.tokenizer = open_clip.get_tokenizer(modelName)
        self.bioMedClip.to(self.device).eval()

    # 💡 Cập nhật mặc định K=1
    def retrieveTopK(self, imageObj, questionText, topK=1): 
        with torch.inference_mode():
            imgRGB = imageObj.convert("RGB") if imageObj.mode != "RGB" else imageObj
            procImg = self.preprocessImage(imgRGB).unsqueeze(0).to(self.device)
            tokens = self.tokenizer([questionText]).to(self.device)
            
            imgFeat = self.bioMedClip.encode_image(procImg)
            txtFeat = self.bioMedClip.encode_text(tokens)
            imgFeat /= imgFeat.norm(dim=-1, keepdim=True)
            txtFeat /= txtFeat.norm(dim=-1, keepdim=True)

            queryVec = self.fusionModel(imgFeat, txtFeat).cpu().numpy().astype('float32')
            distances, indices = self.faissIndex.search(queryVec, topK)
            return [self.metaDataList[idx] for idx in indices[0]], distances[0]

# --- 4. MÔ HÌNH SUY LUẬN INTERNVL RAG ---
class InternVLRagGenerator:
    def __init__(self):
        torch.cuda.empty_cache()
        gc.collect()
        
        print("\n[Hạ tầng] Đang khởi tạo mô hình InternVL3.5-2B (RAG K=1)...")
        modelId = "OpenGVLab/InternVL3_5-2B-Instruct"
        
        self.model = AutoModel.from_pretrained(
            modelId,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto"
        ).eval()
        self.tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)
        print("✅ Đã nạp thành công mô hình InternVL3.5-2B!")

    def processImage(self, pilImg, targetSize=448):
        if pilImg.mode != "RGB": 
            pilImg = pilImg.convert("RGB")
        imageTransform = T.Compose([
            T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])
        return imageTransform(pilImg).unsqueeze(0)

    def generate(self, testImg, testQuest, refContexts, trainDS):
        pixelValuesList = []
        
        # 💡 Prompt cho K=1
        promptText = (
            "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
            "based on an input pathology image and retrieved similar cases.\n"
            "Strictly follow these rules:\n"
            "1. Analyze the provided [Target Case Image] and [Current Question].\n"
            "2. Evaluate the [Retrieved Evidence] from the knowledge base.\n"
            "3. Provide a short, concise, and definitive answer (usually 1 to 5 words).\n"
            "4. Do NOT explain your reasoning.\n\n"
            "[Retrieved Evidence (Top-1)]\n"
            "Below is a similar past case from the clinical database:\n\n"
        )
        
        for i, ctx in enumerate(refContexts):
            try:
                refData = trainDS[int(ctx['id'])]
                if refData.get('image') is None: continue
                    
                rawRefImg = refData['image'].convert("RGB")
                refPixelValues = self.processImage(rawRefImg)
                pixelValuesList.append(refPixelValues)
                
                promptText += f"Case Reference:\n<image>\nQuestion: {refData['question']}\nAnswer: {refData['answer']}\n\n"
            except Exception:
                continue
                
        targetPixelValues = self.processImage(testImg)
        pixelValuesList.append(targetPixelValues)
        
        promptText += (
            f"[Target Case Image]\n<image>\n\n"
            f"Current Question: {testQuest}\n\n"
            "Based on the visual evidence and the clinical context provided above, what is the correct answer for the current case?\n"
            "Final Answer: "
        )
        
        stackedPixelValues = torch.cat(pixelValuesList, dim=0).to(torch.bfloat16).cuda()
        generationConfig = dict(max_new_tokens=15, do_sample=False)
        
        with torch.inference_mode():
            response = self.model.chat(
                tokenizer=self.tokenizer,
                pixel_values=stackedPixelValues,
                question=promptText,
                generation_config=generationConfig
            )
            
        return response.strip()

# --- 5. HÀM THỰC THI CHÍNH ---
def runInternVLRagPathVqaBenchmark(sampleSize=500):
    print("1. Khởi tạo các thư viện đo lường (Metrics)...")
    bleuMetric = evaluate.load("bleu")
    rougeMetric = evaluate.load("rouge")
    bertscoreMetric = evaluate.load("bertscore")
    
    print("2. Tải Retriever và Generator...")
    retriever = PathVqaRetriever()
    slm = InternVLRagGenerator()
    
    print("3. Tải bộ dữ liệu PathVQA...")
    testDS = load_dataset("flaviagiammarino/path-vqa", split="test")
    trainDS = load_dataset("flaviagiammarino/path-vqa", split="train")
    
    totalIndices = len(testDS)
    sampledIndices = random.sample(range(totalIndices), min(sampleSize, totalIndices))
    
    results = {"closed": {"ref": [], "pred": []}, "open": {"ref": [], "pred": []}}
    latencyList = []
    skippedCount = 0
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    print(f"\n🚀 BẮT ĐẦU CHẠY LIGHT-MVQA (INTERNVL3.5-2B + RAG K=1) TRÊN {len(sampledIndices)} MẪU PATHVQA...")
    print("-" * 60)
    
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    startTotalTime = time.time()

    for progressCount, idx in enumerate(sampledIndices):
        try:
            item = testDS[idx]
            if item.get('image') is None:
                skippedCount += 1
                continue
                
            rawImg = item['image'].convert("RGB")
            questionText = str(item['question'])
            
            startTimeSample = time.time()
            
            # Truy xuất Top 1
            contexts, _ = retriever.retrieveTopK(rawImg, questionText, topK=1)
            
            aiAnswerRaw = slm.generate(rawImg, questionText, contexts, trainDS)
            
        except Exception as e:
            print(f"⚠️ Lỗi ngầm tại mẫu {idx}: {e}")
            skippedCount += 1
            continue
            
        latencyList.append(time.time() - startTimeSample)
        
        aiAnswer = aiAnswerRaw.lower().strip()
        groundTruth = str(item['answer']).lower().strip()
        
        if groundTruth in ['yes', 'no']:
            results["closed"]["ref"].append(groundTruth)
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            results["closed"]["pred"].append(predClean)
        else:
            results["open"]["ref"].append(groundTruth)
            results["open"]["pred"].append(aiAnswer)
            
        if (progressCount + 1) % 20 == 0: 
            print(f"Tiến độ: {progressCount + 1}/{len(sampledIndices)} mẫu...")

        del rawImg, contexts, aiAnswerRaw
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    endTotalTime = time.time()
    
    print("\n" + "="*60 + "\n📊 BÁO CÁO THỰC NGHIỆM - LIGHT-MVQA (INTERNVL3.5-2B ON PATHVQA | RAG K=1)\n" + "="*60)
    print(f"📌 Đã quét: {len(sampledIndices)} | Bỏ qua do lỗi: {skippedCount}")
    
    if results["closed"]["ref"]:
        accScore = accuracy_score(results["closed"]["ref"], results["closed"]["pred"])
        f1Weighted = f1_score(results["closed"]["ref"], results["closed"]["pred"], average='weighted', zero_division=0)
        print(f"\n[1. Closed-Ended Classification (Total: {len(results['closed']['ref'])} samples)]")
        print(f"  - Accuracy : {accScore:.4f}")
        print(f"  - F1-Score : {f1Weighted:.4f}")
    
    if results["open"]["ref"]:
        print(f"\n[2. Open-Ended Text Generation (Total: {len(results['open']['ref'])} samples)]")
        predsList = results["open"]["pred"]
        refsList = [[r] for r in results["open"]["ref"]]
        
        bleu1 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=1, smooth=True)['bleu']
        bleu2 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=2, smooth=True)['bleu']
        bleu3 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=3, smooth=True)['bleu']
        rougeL = rougeMetric.compute(predictions=predsList, references=results["open"]["ref"])['rougeL']
        bertResults = bertscoreMetric.compute(predictions=predsList, references=results["open"]["ref"], lang="en")
        
        print(f"  - BLEU-1   : {bleu1:.4f}")
        print(f"  - BLEU-2   : {bleu2:.4f}")
        print(f"  - BLEU-3   : {bleu3:.4f}")
        print(f"  - ROUGE-L  : {rougeL:.4f}")
        print(f"  - BERTScore: {np.mean(bertResults['f1']):.4f}")

    totalDuration = endTotalTime - startTotalTime
    if latencyList:
        print("\n[3. Computational Efficiency Measurements]")
        print(f"  - Inference Latency (Độ trễ trung bình) : {np.mean(latencyList):.2f} s/mẫu")
        print(f"  - Throughput (Tốc độ xử lý)             : {len(latencyList) / totalDuration:.2f} mẫu/giây")
    
    if torch.cuda.is_available():
        print("\n[4. Multi-GPU Memory Profile]")
        for gpuId in range(torch.cuda.device_count()):
            vramMB = torch.cuda.max_memory_allocated(device=gpuId) / (1024 ** 2)
            print(f"  - Peak Memory Usage on GPU {gpuId}        : {vramMB:.2f} MB")
    print("="*60)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == "__main__":
    runInternVLRagPathVqaBenchmark(sampleSize=500)

1. Khởi tạo các thư viện đo lường (Metrics)...


2. Tải Retriever và Generator...


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

... (40 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

  - BLEU-1   : 0.2848
  - BLEU-2   : 0.2098
  - BLEU-3   : 0.1715
  - ROUGE-L  : 0.3853
  - BERTScore: 0.9020

[3. Computational Efficiency Measurements]
  - Inference Latency (Độ trễ trung bình) : 1.83 s/mẫu
  - Throughput (Tốc độ xử lý)             : 0.45 mẫu/giây

[4. Multi-GPU Memory Profile]
  - Peak Memory Usage on GPU 0        : 4170.25 MB
  - Peak Memory Usage on GPU 1        : 2328.53 MB
